In [1]:
Output = c("/Users/alexis/Library/CloudStorage/OneDrive-UniversityofNorthCarolinaatChapelHill/CEMALB_DataAnalysisPM/Projects/P1014. Nanoparticles Respiratory Tract/P1014.3. Analyses/P1014.3.1. Group Distribution Analysis/Output")
cur_date = "021026"

library(readxl)
library(openxlsx)
library(tidyverse)
library(reshape2)
library(rlang)
library(limma)

# reading in files
protein_df = data.frame(read_excel("Input/Protein_Data_020926.xlsx", sheet = 2))[,4:13]
protein_df$Dose = as.numeric(protein_df$Dose)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.1.0     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attaching package: ‘reshape2’


The following object is masked from ‘package:tidyr’:

    smiths



Attaching package: ‘rlang’


The following objects are masked from ‘package:purrr’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice




In [2]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03001,Q07011,4-1BB,2.002402,1.586117
2,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03541,Q15109,AGER,3.425807,2.145941
3,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.178579,1.668382
4,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL04271,P31749,AKT1,2.214532,1.684609
5,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL03101,Q15389,Angiopoietin-1,2.611234,1.852492
6,1,NP1-Rhodamine B,NP1-Rhodamine B_1_0.05,0.05,R,nEL09731,P05089,ARG1,3.369304,2.127404


In [3]:
# only interested in centrifuged samples
protein_df = protein_df %>%
    filter(Separation == 'C')

we are interested in comparing NP1_Rhodamine B and NP2 with Vehicle control. For NP1 and NP1_Rhodamine B, two doses are available. 
Let's answer this with a 

Now on the data analysis side, we are interested in dose-dependent effect of NP1 and NP1_Rhodamine B. A comparison between NP1 and NP1_Rhodamine B will also be informative. Ultimately, the effect of NP2 and NP2_Rhodamine B (with only n=1) at the higher dose will be very informative. Comparison between NP1 and NP2 at the higher dose will be informative too.


## Is there a statistically significant difference in protein expression in respiratory tract cells based on treatment? 

We'll linear modeling to compare, NP1 (at both doses), NP1-Rhodamine B (at both doses) and NP2 to controls.

In [4]:
unique(protein_df$Treatment)
unique(protein_df$Dose)

[1] "Control"         "NP1"             "NP1-Rhodamine B" "NP2_Rhodamine B"
[5] "NP2"

[1]   NA 0.10 0.05

In [5]:
# creating a comparison df to iterate through
comparison_df = data.frame(#Group1 = c(rep(c("Control"), times = 6)),
                           Tx = c(unique(protein_df$Treatment)[2:5], unique(protein_df$Treatment)[2:3]),
                           Dose = c(0.05,0.05,0.1,0.1,0.1,0.1))

comparison_df

Tx,Dose
<chr>,<dbl>
NP1,0.05
NP1-Rhodamine B,0.05
NP2_Rhodamine B,0.10
NP2,0.10
NP1,0.10
NP1-Rhodamine B,0.10


In [10]:
wilcoxon_values = function(df, comp_df){
    # """
    # Running t tests after filtering for set, treatment, exposure, and protein using a loop. 
    # Ultimately using this test to compare different treatment groups to controls.

    # :param: subsetted dataframe, empty dataframe
    # :output: a dataframe containing the set, treatment, exposure, protein, u stat, p value, p adj

    # """
    proteins = unique(df$Protein_Name)
    
    values_df = data.frame()
    # iterating through each tx, protein

    for(i in 1:length(proteins)){
        for(j in 1:length(comp_df$Tx)){
            # unexposed df
            unexposed_df = df %>%
                filter(Protein_Name == proteins[i], Treatment == "Control")
            # exposed df
            exposed_df = df %>%
                filter(Protein_Name == proteins[i], Treatment == comp_df$Tx[j], 
                       Dose == comp_df$Dose[j])

            # wilcoxon test
            wilcox_test = wilcox.test(unexposed_df$Conc_pslog2, exposed_df$Conc_pslog2)
            
            # calculating FC to get directionality
            FC = log2(mean(exposed_df$Conc)/mean(unexposed_df$Conc))

            # contains the treatments compared, dose, protein, test statistic, and p value
            values_vector = cbind("Control", comp_df$Tx[j], comp_df$Dose[j], proteins[i], FC, wilcox_test$statistic, 
                                  wilcox_test$p.value)
            values_df = rbind(values_df, values_vector)
            }
        }

    
    # adding col names
    colnames(values_df) = c("Group1", "Group2", "Dose", "Protein_Name", "log2FC", "Statistic", "P Value")
    
    
   # calculating padj values
    values_df = values_df %>%
        group_by(Group2, Dose) %>%
        mutate(`P Value` = as.numeric(`P Value`),
               log2FC = as.numeric(log2FC),
           `P Adj` =  p.adjust(`P Value`, method = "fdr")) 
    
    return(values_df)
}

# calling fn
control_tx_wilcox_df = wilcoxon_values(protein_df, comparison_df)

head(control_tx_wilcox_df)

Group1,Group2,Dose,Protein_Name,log2FC,Statistic,P Value,P Adj
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
Control,NP1,0.05,4-1BB,-0.020094658,8,0.2,0.6409091
Control,NP1-Rhodamine B,0.05,4-1BB,-0.009828542,7,0.4,0.9808696
Control,NP2_Rhodamine B,0.1,4-1BB,-0.041170479,3,0.5,1.0000000
Control,NP2,0.1,4-1BB,-0.003089979,6,0.7,0.8657895
Control,NP1,0.1,4-1BB,0.012048215,3,0.7,0.9444976
Control,NP1-Rhodamine B,0.1,4-1BB,-0.012078402,8,0.2,0.7421053


In [12]:
control_tx_wilcox_df %>%
    filter(Protein_Name == 'TNFRSF11B')#`P Adj` < 0.1)

Group1,Group2,Dose,Protein_Name,log2FC,Statistic,P Value,P Adj
<chr>,<chr>,<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>
Control,NP1,0.05,TNFRSF11B,-0.4358076,9,0.1,0.6000000
Control,NP1-Rhodamine B,0.05,TNFRSF11B,-0.1707675,9,0.1,0.8812500
Control,NP2_Rhodamine B,0.1,TNFRSF11B,-0.5372194,3,0.5,1.0000000
Control,NP2,0.1,TNFRSF11B,-0.6235698,9,0.1,0.3481481
Control,NP1,0.1,TNFRSF11B,-0.6635312,9,0.1,0.5423077
Control,NP1-Rhodamine B,0.1,TNFRSF11B,-0.4415180,9,0.1,0.6130435


There are no signficant differences in protein expression between controls and treatment groups regardless of dose.

In [15]:
head(protein_df)

,Subject_No,Treatment,Sample_ID,Dose,Separation,ELISA_ID,UniProt_ID,Protein_Name,Conc,Conc_pslog2
,<dbl>,<chr>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
1,2,Control,Control_2_NA,NA,C,nEL03001,Q07011,4-1BB,2.000806,1.585350
2,2,Control,Control_2_NA,NA,C,nEL03541,Q15109,AGER,3.394482,2.135693
3,2,Control,Control_2_NA,NA,C,nEL01701,Q9UNG2,AITRL (GITR Ligand),2.122618,1.642756
4,2,Control,Control_2_NA,NA,C,nEL04271,P31749,AKT1,2.183432,1.670583
5,2,Control,Control_2_NA,NA,C,nEL03101,Q15389,Angiopoietin-1,2.722051,1.896098
6,2,Control,Control_2_NA,NA,C,nEL09731,P05089,ARG1,3.349175,2.120742


In [4]:
# converting to a wide format to run limma
wide_protein_matrix = protein_df %>%
    select(Sample_ID, UniProt_ID, Conc_pslog2) %>%
    pivot_wider(names_from = Sample_ID, values_from = Conc_pslog2) %>%
    column_to_rownames("UniProt_ID") %>%
    as.matrix()

head(wide_protein_matrix)

,Control_2_NA,NP1_3_0.1,NP1-Rhodamine B_5_0.1,NP1_6_0.05,NP1-Rhodamine B_8_0.05,Control_10_NA,NP1_11_0.1,NP1-Rhodamine B_13_0.1,NP1_14_0.05,NP2_Rhodamine B_15_0.1,NP1-Rhodamine B_16_0.05,Control_17_NA,NP2_18_0.1,NP1_20_0.1,NP1-Rhodamine B_21_0.1,NP2_22_0.1,NP1_24_0.05,NP1-Rhodamine B_25_0.05,NP2_26_0.1
Q07011,1.585350,1.569602,1.569492,1.572613,1.576097,1.583870,1.605838,1.582235,1.554672,1.554169,1.582363,1.575114,1.584561,1.592782,1.568494,1.573283,1.576927,1.566238,1.580319
Q15109,2.135693,2.138565,2.141932,2.143017,2.133122,2.073869,2.129192,2.129461,2.148347,2.137005,2.125730,2.138415,2.145104,2.115151,2.133203,2.128770,2.139980,2.125594,2.136470
Q9UNG2,1.642756,1.654883,1.636094,1.646805,1.645349,1.674437,1.661696,1.657174,1.650657,1.654289,1.649807,1.629017,1.652983,1.664520,1.664310,1.658886,1.660484,1.642435,1.642868
P31749,1.670583,1.680710,1.675403,1.663189,1.665997,1.663760,1.660594,1.666499,1.667402,1.663113,1.683948,1.649666,1.686324,1.671090,1.677939,1.693824,1.684437,1.676334,1.684229
Q15389,1.896098,1.773931,1.819644,1.803776,1.843614,1.877910,1.776046,1.818144,1.791555,1.709945,1.860328,1.851314,1.684969,1.783841,1.817430,1.677256,1.810487,1.823491,1.655942
P05089,2.120742,2.121573,2.120719,2.109544,2.123041,2.113760,2.128535,2.126708,2.149856,2.117535,2.133877,2.121512,2.109153,2.133167,2.131711,2.129999,2.132945,2.111452,2.146192


In [5]:
# creating a sample info df
sample_info_df = protein_df[,2:4] %>%
    unique() %>%
    # ensuring the sample ids are in the same order as the previous df
    arrange(match(Sample_ID, colnames(wide_protein_matrix)))

head(sample_info_df)

,Treatment,Sample_ID,Dose
,<chr>,<chr>,<dbl>
1,Control,Control_2_NA,0.00
2,NP1,NP1_3_0.1,0.10
3,NP1-Rhodamine B,NP1-Rhodamine B_5_0.1,0.10
4,NP1,NP1_6_0.05,0.05
5,NP1-Rhodamine B,NP1-Rhodamine B_8_0.05,0.05
6,Control,Control_10_NA,0.00


In [10]:
design = model.matrix(~ Treatment + Dose, data = sample_info_df)
linear_fit = lmFit(wide_protein_matrix, design)
limma_fit = eBayes(linear_fit)

 topTable(limma_fit) %>% #, coef = "GroupTreated", number = Inf, sort.by = "P") 
rownames_to_column("UniProt_ID")

Removing intercept from test coefficients



UniProt_ID,TreatmentNP1,TreatmentNP1.Rhodamine.B,TreatmentNP2,TreatmentNP2_Rhodamine.B,Dose,AveExpr,F,P.Value,adj.P.Val
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
P10909,-0.228597401,-0.131201250,-0.30628528,-0.29093662,-2.16588119,1.649711,329.99254,1.429920e-18,4.003775e-16
O00300,-0.131216962,0.036666773,-0.09533757,-0.03786706,-3.43723747,1.706025,274.74035,8.774602e-18,1.228444e-15
Q9H2A7,-0.015556927,0.046884967,-0.25650281,-0.12719368,-1.00298329,2.772232,179.14693,5.914084e-16,5.519812e-14
P01375,-0.011066112,-0.018877635,-0.19298967,-0.15737873,0.18121291,1.897630,109.24271,7.277015e-14,5.093910e-12
Q15389,-0.049113744,-0.008611438,-0.15431298,-0.11709073,-0.48072046,1.793459,81.73528,1.173276e-12,6.570346e-11
Q99988,0.071537098,0.087131367,-0.05551600,-0.03777834,-0.08769099,2.611512,60.83389,1.903047e-11,8.880887e-10
P10646,0.068444262,0.020007437,-0.19629745,-0.19111882,0.26047991,2.102552,48.86597,1.449031e-10,5.796125e-09
P19875,-0.004116102,-0.003701589,-0.13678105,-0.11907606,0.08401045,2.800117,36.19366,2.194895e-09,7.682134e-08
P42830,-0.039675902,0.001067040,-0.16717463,-0.17692665,-0.06168329,2.252989,23.48958,9.288706e-08,2.821647e-06


In [8]:
limma_fit$coefficients

,(Intercept),TreatmentNP1,TreatmentNP1-Rhodamine B,TreatmentNP2,TreatmentNP2_Rhodamine B,Dose
Q07011,1.581445,-0.0175887920,-0.022174637,-0.021901468,-0.047119670,0.198442986
Q15109,2.115992,0.0267883663,0.022586625,0.030218339,0.030441695,-0.094291456
Q9UNG2,1.648737,-0.0030140165,-0.010326685,-0.011537540,-0.008827865,0.143797712
P31749,1.661337,0.0121683393,0.015284728,0.029812771,0.004800519,-0.030237711
Q15389,1.875107,-0.0491137442,-0.008611438,-0.154312984,-0.117090731,-0.480720457
P05089,2.118671,0.0101740982,0.005488835,0.009210383,-0.001702143,0.005660083
P20160,1.922367,0.0028467332,-0.004156750,-0.008175644,0.008178716,-0.101224052
P10415,1.669988,0.0159048596,0.014506337,0.026592344,0.018409462,-0.056388819
Q02223,1.399469,0.0101106100,0.014890424,0.011410571,0.005236876,-0.026418020
P78410,1.606170,0.0281686975,0.023606200,0.036271561,0.021500995,-0.038660697


In [11]:
limma_fit$F

[1]  47932.110  55560.288  51012.179  69810.934  51636.539  81213.239
  [7]  90332.243  28279.820  25570.225  22261.353  57583.809  36956.776
 [13]  19808.856  86157.669   7798.756  43214.619  29419.727  17949.510
 [19]  38818.805  46026.309  24343.797  32505.931   5993.205  22821.121
 [25]  58807.848   7201.298  41033.185   6290.605  26275.335  54882.988
 [31]  85441.178   7500.943  38749.274  14403.150  35351.789   4892.135
 [37]  19400.683  68388.225  14050.817  44590.184  38196.209 182071.723
 [43]   4718.153  14248.117  40420.082  28013.479  84278.535  34667.145
 [49]  65570.905  21856.185  43550.252  38671.363  22632.463  72946.064
 [55]  17437.389  19681.797  55271.691  56176.194  71616.428  66384.004
 [61] 112161.683   7325.641  26546.906  34274.882  42048.536  95513.820
 [67]  33830.586  27709.105  19800.744  19288.173  21212.646  62078.277
 [73]  80638.329  59621.704  26859.125  62095.776  32037.300  48695.486
 [79]  91654.816  19444.849  31564.308  59099.877  60048.067  79974.515
 [85]  15912.565  46009.195 160771.617  14554.055   8557.568  11485.895
 [91]  21974.943  16714.337  77370.944  31634.661  89302.633  45219.114
 [97]  22511.811  19691.795  11292.343  15196.216 130348.711  17206.694
[103]  47395.308  69022.466  35145.728  35531.700  17699.797  52675.340
[109]  67018.527  19628.499  46605.285  37328.901  23090.640  36631.608
[115]  31854.446  56463.990  33067.018  85425.429  14647.462  21624.015
[121]  73401.572   2520.059 113818.746  27504.646  13505.508  25671.065
[127] 123289.147  10574.810  48709.906  17418.540  25594.196  37917.847
[133]  65954.020  14440.033  30557.554  16975.187  46114.114  61259.278
[139]  24657.090  15096.584  78684.674  71518.446  32065.810  13855.715
[145]  30106.864  17970.374  62299.736  31800.416  18714.812  42534.193
[151]  40325.020  41315.810  31865.046  49151.779  49970.132  47774.520
[157]   1990.406  39723.856  94422.178  44428.409  48472.480  77848.242
[163]  19464.890  24923.095  33290.354  47174.231  27270.523  35142.148
[169]  15155.457  25290.666  40261.791  11171.360  48030.975  59710.985
[175]  46381.450  28819.379  15502.642  46000.317  68612.357  60428.685
[181]  32809.140  27033.202  26610.379  25932.407  31117.620  59587.751
[187]  49085.222  52936.406  29514.239  58831.974  93202.061  28981.637
[193]  16031.446  39775.641 267531.267  44370.917 118123.796 113561.282
[199]  31424.671  57351.660  23917.671  11575.814  54699.080  77567.617
[205]  43184.774  80785.203  77067.222  37120.832  37819.407  67305.780
[211]  36429.379  32188.624  49397.548  73876.646 168805.149  65481.845
[217]  36710.070  34606.068  32007.324  27443.137  75418.527  20657.888
[223]  45070.835  20786.875  58630.523  28440.858  64648.334 105686.632
[229]  32735.121 109582.658  19096.595  31811.638  28542.482  40971.802
[235]  51985.301  18347.922  35118.070  52657.836  46193.262  54204.137
[241]  60939.226   6985.662  20178.099  20146.640  68506.292  47628.790
[247]  45962.318  24129.115  10274.567  75822.320  18110.890 208154.944
[253]  24431.409  36312.993  71508.157   8510.984  43311.978  36007.079
[259]  25814.739  25447.676  48986.303  30149.183  60276.680  58843.826
[265]  17910.953  31577.829  19863.497  33995.364  88798.514  33243.747
[271]  17883.356  67607.695 170267.539 138473.027 200766.141  77818.996
[277]  14887.474  31227.766  15747.274  57881.753